In [ ]:
import sys
import os
import matplotlib.pyplot as plt

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.LoadupSamples import LoadupSamples
from src.predictionModule.FilterSamples import FilterSamples
from src.predictionModule.MachineModels import MachineModels

import numpy as np
import pandas as pd
import datetime
import optuna
import random
import torch
import copy

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_samples, silhouette_score

import logging
formatted_date = datetime.datetime.now().strftime("%d%b%y_%H%M").lower()

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(fmt="%(asctime)s - %(message)s")
handler.setFormatter(formatter)
#if not logger.hasHandlers():
#    logger.addHandler(handler)
#else:
#    logger.handlers[:] = [handler]

#Output File handler
formatted_str = f"notebook-stomp-{formatted_date}"
file_handler = logging.FileHandler(f"{formatted_str}.log", mode="w")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Usage
logger.setLevel(logging.INFO)
logger.info("This will print to the notebook's output cell")

In [2]:
params = {
    "idxAfterPrediction": 5,
    'timesteps': 90,
    'target_option': 'last',
    "LoadupSamples_time_scaling_stretch": False,
    "LoadupSamples_time_inc_factor": 1,

    "FilterSamples_q_up": 0.6,
    
    "FilterSamples_cat_over20": True,
    "FilterSamples_cat_posOneYearReturn": False,
    "FilterSamples_cat_posFiveYearReturn": False,

    "LSTM_val_split": 0.1,
}

In [3]:
timegroup = "group_regOHLCV_over5years"
treegroup = "group_debug"

eval_date = datetime.date(year=2025, month=7, day=13)
evaldates = [eval_date - datetime.timedelta(days=i) for i in range(1, 6)]
start_train_date = datetime.date(year=2016, month=1, day=1)
split_Date = datetime.date(year=2025, month=1, day=1)
ls = LoadupSamples(
    train_start_date=start_train_date,
    test_dates=evaldates,
    treegroup=treegroup,
    timegroup=timegroup,
    params=params,
)
ls.load_samples(main_path = "../src/featureAlchemy/bin/")
ls.split_dataset(
    start_date=start_train_date,
    last_train_date=split_Date,
    last_test_date=eval_date
)
fs_pre = FilterSamples(
    Xtree_train = ls.train_Xtree, 
    ytree_train = ls.train_ytree, 
    treenames   = ls.featureTreeNames,
    Xtree_test  = ls.test_Xtree,  
    ytree_test  = ls.test_ytree,
    meta_train  = ls.meta_pl_train, 
    meta_test   = ls.meta_pl_test, 
    params      = params
)
mask_train_pre, mask_test_pre = fs_pre.categorical_masks()
ls.apply_masks(mask_train_pre, mask_test_pre)

In [4]:
Xtree_train = ls.train_Xtree
ytree_train = ls.train_ytree
Xtree_test  = ls.test_Xtree
ytree_test  = ls.test_ytree

Xtime_train = ls.train_Xtime
ytime_train = ls.train_ytime
Xtime_test  = ls.test_Xtime
ytime_test  = ls.test_ytime

treenames   = ls.featureTreeNames
timenames   = ls.featureTimeNames
meta_train  = ls.meta_pl_train
meta_test   = ls.meta_pl_test

dates_tr = meta_train['date'].unique().sort()
dates_te = meta_test['date'].unique().sort()

from src.common.DataFrameTimeOperations import DataFrameTimeOperations as dfta
dates_tr_idx = dfta(meta_train, 'date').getNextLowerOrEqualIndices(dates_tr)
dates_te_idx = dfta(meta_test, 'date').getNextLowerOrEqualIndices(dates_te)

assert not any([i == -1 for i in dates_tr_idx])
assert not any([i == -1 for i in dates_te_idx])

In [5]:
# ---- knobs (use existing globals if present) ----
device = "cuda" if torch.cuda.is_available() else "cpu"
n_splits = 500
n_test_days = 60

max_training_days = 365*3

N = len(dates_tr)
lo = max_training_days - 1                      # min pivot (last train index)
hi = N - n_test_days - 2                      # max pivot
eligible = list(range(lo, hi + 1))
assert len(eligible) >= n_splits, f"Too few eligible pivots ({len(eligible)}) for n_splits={n_splits}"
pivots = sorted(random.sample(eligible, n_splits))

In [6]:
assert "Xtime_train" in globals() and "ytree_train" in globals(), "Need Xtime_train/ytree_train"
nS, nT, nF = Xtime_train.shape
def geometric_mean_safe(arr):
    arr = np.asarray(arr, dtype=float)
    minv = np.min(arr) if arr.size else 0.0
    shift = -minv + 1e-9 if minv <= 0 else 0.0
    return float(np.exp(np.mean(np.log(arr + shift)))) if arr.size else np.nan

def metric(arr):
    """Custom cluster score function."""
    gm = geometric_mean_safe(arr)
    return gm - 1

def make_design(X, t_win, feat_idx): 
    feat_idx = np.atleast_1d(feat_idx)
    Xw = X[:, -t_win:, feat_idx]
    return Xw.reshape(Xw.shape[0], -1)

def make_design2(X, t_win, feat_idx, amp1=2.0):
    feat_idx = np.atleast_1d(feat_idx)
    Xw = X[:, -(t_win+1):, :][:, :, feat_idx]                          # (N, t_win+1, F)

    # Percent change along time: (x_t / x_{t-1} - 1)
    eps = 1e-9
    prev = Xw[:, :-1, :]
    curr = Xw[:, 1:, :]
    pct = (curr / (prev + eps)) - 1.0                              # (N, t_win, F)

    # Weights that increase for later timesteps
    Tm1 = pct.shape[1]
    w = np.linspace(0.0, 1.0, num=Tm1, dtype=pct.dtype)**amp1       # 1, 2, ..., t_win
    s = w.sum()
    if s > 0:
        w = w / s

    pct_weighted = pct * w[None, :, None]                          # broadcast over time
    return pct_weighted.reshape(pct.shape[0], -1)

def _score_once_lstm(
    t_win: int,
    k: int,
    f_idcs: list[int],
    Xtr: np.ndarray,
    ytr_tree: np.ndarray,
    ytr_time: np.ndarray,
    Xte: np.ndarray,
    yte_tree: np.ndarray,
    yte_time: np.ndarray,
    mm: MachineModels,
    quantile_val: float = 0.9,
    do_transform: bool = False,
    device: str = "cpu",
    min_cluster_train: int = 50,
) -> float:
    """
    Cluster train window, train one LSTM per cluster, pick cluster with lowest RMSE,
    predict on the paired test samples in that cluster, and select the highest
    predicted entries via a quantile threshold.
    """

    if k >= Xtr.shape[0]:
        raise ValueError(f"Too many clusters k={k} for n_samples={Xtr.shape[0]}")

    # design matrices
    Xd_tr = make_design2(Xtr, t_win, f_idcs)
    Xd_te = make_design2(Xte, t_win, f_idcs)

    if do_transform:
        scaler = StandardScaler().fit(Xd_tr)
        Xd_tr = scaler.transform(Xd_tr)
        Xd_te = scaler.transform(Xd_te)

    n_feat = len(np.atleast_1d(f_idcs))
    Xseq_tr = Xd_tr.reshape(-1, t_win, n_feat)
    Xseq_te = Xd_te.reshape(-1, t_win, n_feat)

    km = KMeans(n_clusters=k, n_init="auto", random_state=0, init="k-means++").fit(Xd_tr)
    lab_tr = km.labels_
    lab_te = km.predict(Xd_te)

    # Silhouette scores
    sil_samples = silhouette_samples(Xd_tr, lab_tr)

    cluster_labels = np.arange(k)
    models = np.array([None] * k)
    rmses = np.ones(k) * np.inf
    mean_vals_te = np.ones(k) * metric(1.0)
    mean_vals_tr = np.ones(k) * metric(1.0)
    mean_quant_te_up = np.ones(k) * metric(1.0)
    mean_quant_te_do = np.ones(k) * metric(1.0)
    mean_silhouette = np.ones(k) * metric(1.0)
    for c in range(k):
        mask_tr = (lab_tr == c)
        mask_te = (lab_te == c)
        if mask_tr.sum() < min_cluster_train:
            logger.info(f"[LSTM] cluster {c}: skipped (train size {mask_tr.sum()} < {min_cluster_train})")
            continue

        Xc_tr = Xseq_tr[mask_tr]
        Xc_te = Xseq_te[mask_te]
        yc_tr = ytr_time[mask_tr]
        yc_te = yte_time[mask_te]

        try:
            logger.disabled = True
            model_c, info = mm.run_LSTM_torch(
                X_train=Xc_tr,
                y_train=yc_tr,
                X_test=None,
                y_test=None,
                device=device,
                logger_disabled=True,
            )
            logger.disabled = False
            val_rmse = float(info.get("val_rmse", metric(1.0)))

            preds = mm.predict_LSTM_torch(model_c, Xc_te, device=device)
            thr_hi = np.quantile(preds, quantile_val)
            thr_lo = np.quantile(preds, 1 - quantile_val)
            mask_up = preds >=  thr_hi
            mask_do = preds <=  thr_lo
            y_selected_up = yte_tree[mask_te][mask_up]
            y_selected_do = yte_tree[mask_te][mask_do]
            y_selected_up = y_selected_up[np.isfinite(y_selected_up)]
            y_selected_do = y_selected_do[np.isfinite(y_selected_do)]

            models[c] = model_c
            rmses[c] = val_rmse
            mean_vals_tr[c] = metric(ytr_tree[mask_tr])
            mean_vals_te[c] = metric(yte_tree[mask_te])
            mean_quant_te_up[c] = metric(y_selected_up) if y_selected_up.size > 0 else metric(1.0)
            mean_quant_te_do[c] = metric(y_selected_do) if y_selected_do.size > 0 else metric(1.0)
            mean_silhouette[c] = np.mean(sil_samples[mask_tr])

        except Exception as e:
            logger.disabled = False
            logger.warning(f"[LSTM] cluster {c} failed - {e}")
            continue
        finally:
            logger.disabled = False

    for c in cluster_labels:
        logger.info(
            f"[LSTM] cluster {c}: \n"
            f"  train={np.sum(lab_tr == c)}\n"
            f"  rmse={rmses[c]:.6f}\n"
            f"  mean_tr={mean_vals_tr[c]:.6f}\n"
            f"  mean_te={mean_vals_te[c]:.6f}\n"
            f"  mean_quant_te_up={mean_quant_te_up[c]:.6f}\n"
            f"  mean_quant_te_do={mean_quant_te_do[c]:.6f}\n"
            f"  mean_silhouette={mean_silhouette[c]:.6f}\n"
        )

    return {
        "rmses": rmses,
        "mean_vals_tr": mean_vals_tr,
        "mean_vals_te": mean_vals_te,
        "mean_quant_te_up": mean_quant_te_up,
        "mean_quant_te_do": mean_quant_te_do,
        "mean_silhouette": mean_silhouette,
        "models": models,
    }

In [7]:
opt_params = copy.deepcopy(params)
opt_params = {
    "t_win": 20,
    "k": 8,
    "n_training_days": 1000,
    "do_transform": False,
    "f_idcs_cat": 0,
    "quantile_val": 0.8,
}
opt_params["idxAfterPrediction"] = 5
opt_params["LoadupSamples_time_inc_factor"] = 61
opt_params["LSTM_units"] = 8
opt_params["LSTM_num_layers"] = 1
opt_params["LSTM_learning_rate"] = 1e-4
opt_params["LSTM_epochs"] = 2
opt_params["LSTM_l1"] = 1e-2
opt_params["LSTM_l2"] = 1e-2
opt_params["LSTM_dropout"] = 0.05
opt_params["LSTM_inter_dropout"] = 0.05
opt_params["LSTM_recurrent_dropout"] = 0.05
opt_params["LSTM_conv1d_kernel_size"] = 3
opt_params["is_single_feature"] = False

t_win = opt_params["t_win"]
k = opt_params["k"]
n_training_days = opt_params["n_training_days"]
do_transform = opt_params["do_transform"]
f_idcs_cat = opt_params["f_idcs_cat"]
quantile_val = opt_params["quantile_val"]

if f_idcs_cat == 0:       f_idcs = [0]
elif f_idcs_cat == 1:     f_idcs = [1]
elif f_idcs_cat == 2:     f_idcs = [0, 1]

time_factor = opt_params["LoadupSamples_time_inc_factor"]
ytime_train = np.tanh((ytree_train - 1.0) * time_factor) / 2.0 + 0.5
ytime_test = np.tanh((ytree_test - 1.0) * time_factor) / 2.0 + 0.5

logger.info(f"Params: {opt_params}")
splits_result = []
mm = MachineModels(params=opt_params)
for i in range(n_splits):
    p = pivots[i]
    # example: get date bounds if needed
    s_tr_l = dates_tr_idx[p - n_training_days+1]
    s_tr_u = dates_tr_idx[p + 1]-1
    s_te_l = dates_tr_idx[p + 1]
    s_te_u = dates_tr_idx[p + 1 + n_test_days]-1
    s_tr = slice(s_tr_l, s_tr_u)
    s_te = slice(s_te_l, s_te_u)

    Xtr, ytr_time, ytr_tree = Xtime_train[s_tr], ytime_train[s_tr], ytree_train[s_tr]
    Xte, yte_time, yte_tree = Xtime_train[s_te], ytime_train[s_te], ytree_train[s_te]
    try:
        res = _score_once_lstm(t_win, k, f_idcs, Xtr, ytr_tree, ytr_time, Xte, yte_tree, yte_time, mm,
            device=device, do_transform=do_transform, quantile_val=quantile_val)
    except Exception as e:
        logger.info(f"Exception during scoring: {e}")
        continue
    splits_result.append(res)

In [8]:
mv_tr  = []
mv_te  = []
mqu    = []
mqd    = []
msil   = []
rmses  = []
for split_idx, res in enumerate(splits_result):
    if res is None:
        continue

    mv_tr_res  = res.get("mean_vals_tr",  np.nan)
    mv_te_res  = res.get("mean_vals_te",  np.nan)
    mqu_res    = res.get("mean_quant_te_up", np.nan)
    mqd_res    = res.get("mean_quant_te_do", np.nan)
    msil_res   = res.get("mean_silhouette",  np.nan)
    rmses_res  = res.get("rmses", res.get("rmse", np.nan))

    mv_tr.append(mv_tr_res)
    mv_te.append(mv_te_res)
    mqu.append(mqu_res)
    mqd.append(mqd_res)
    msil.append(msil_res)
    rmses.append(rmses_res)

In [9]:
df_mv_tr_res = pd.DataFrame(mv_tr)
df_mv_te_res = pd.DataFrame(mv_te)
df_mqu_res   = pd.DataFrame(mqu)
df_mqd_res   = pd.DataFrame(mqd)
df_msil_res  = pd.DataFrame(msil)
df_rmses_res = pd.DataFrame(rmses)

np_mv_tr_res = df_mv_tr_res.to_numpy()
np_mv_te_res = df_mv_te_res.to_numpy()
np_mqu_res   = df_mqu_res.to_numpy()
np_mqd_res   = df_mqd_res.to_numpy()
np_msil_res  = df_msil_res.to_numpy()
np_rmses_res = df_rmses_res.to_numpy()

In [10]:
# stack to find rows with NaNs
stacked = np.hstack([np_mv_tr_res, np_mv_te_res, np_mqu_res, np_mqd_res, np_msil_res, np_rmses_res])
mask = ~(np.isnan(stacked).any(axis=1) | np.isinf(stacked).any(axis=1))

# print rows with NaNs
logger.info("Removed rows:")
logger.info(len(stacked[~mask]))

# filter each array
npclean_mv_tr_res = np_mv_tr_res[mask]
npclean_mv_te_res = np_mv_te_res[mask]
npclean_mqu_res   = np_mqu_res[mask]
npclean_mqd_res   = np_mqd_res[mask]
npclean_msil_res  = np_msil_res[mask]
npclean_rmses_res = np_rmses_res[mask]

In [11]:
col_tr_res_idx = np.argmax(np_mv_tr_res, axis=1)
col_rmse_idx = np.argmin(np_rmses_res, axis=1)
col_msil_res = np.argmax(np_msil_res, axis=1)
col_msilmin_idx = np.argmin(np_msil_res, axis=1)

idx = np.arange(np_mv_te_res.shape[0])
tr2te   = np.mean(np_mv_te_res[idx, col_tr_res_idx])
rmse2qu = np.mean(np_mqu_res[idx, col_rmse_idx])
rmse2qd = np.mean(np_mqd_res[idx, col_rmse_idx])
sil2qu  = np.mean(np_mqu_res[idx, col_msil_res])
sil2qd  = np.mean(np_mqd_res[idx, col_msil_res])
sil2te  = np.mean(np_mv_te_res[idx, col_msil_res])
silmin2te = np.mean(np_mv_te_res[idx, col_msilmin_idx])
silmin2qu = np.mean(np_mqu_res[idx, col_msilmin_idx])
silmin2qd = np.mean(np_mqd_res[idx, col_msilmin_idx])

logger.info(f"Mean all {np.mean(np_mv_te_res.mean())}: ")
logger.info(f"Mean train-to-test metric: {tr2te:.6f}")
logger.info(f"Mean RMSE to quantile up metric: {rmse2qu:.6f}")
logger.info(f"Mean RMSE to quantile down metric: {rmse2qd:.6f}")
logger.info(f"Mean silhouette to quantile up metric: {sil2qu:.6f}")
logger.info(f"Mean silhouette to quantile down metric: {sil2qd:.6f}")
logger.info(f"Mean silhouette to test metric: {sil2te:.6f}")
logger.info(f"Mean silhouette min to test metric: {silmin2te:.6f}")
logger.info(f"Mean silhouette min to quantile up metric: {silmin2qu:.6f}")
logger.info(f"Mean silhouette min to quantile down metric: {silmin2qd:.6f}")

#Cleaned version
colclean_tr_res_idx = np.argmax(npclean_mv_tr_res, axis=1)
colclean_rmse_idx = np.argmin(npclean_rmses_res, axis=1)
colclean_msil_res = np.argmax(npclean_msil_res, axis=1)
colclean_msilmin_idx = np.argmin(npclean_msil_res, axis=1)

idx_cl = np.arange(npclean_mv_te_res.shape[0])
tr2te_cl   = np.mean(npclean_mv_te_res[idx_cl, colclean_tr_res_idx])
rmse2qu_cl = np.mean(npclean_mqu_res[idx_cl, colclean_rmse_idx])
rmse2qd_cl = np.mean(npclean_mqd_res[idx_cl, colclean_rmse_idx])
sil2qu_cl  = np.mean(npclean_mqu_res[idx_cl, colclean_msil_res])
sil2qd_cl  = np.mean(npclean_mqd_res[idx_cl, colclean_msil_res])
sil2te_cl  = np.mean(npclean_mv_te_res[idx_cl, colclean_msil_res])
silmin2te_cl = np.mean(npclean_mv_te_res[idx_cl, colclean_msilmin_idx])
silmin2qu_cl = np.mean(npclean_mqu_res[idx_cl, colclean_msilmin_idx])
silmin2qd_cl = np.mean(npclean_mqd_res[idx_cl, colclean_msilmin_idx])

logger.info("After removing NaNs and Infs:")
logger.info(f"  Mean all {np.mean(npclean_mv_te_res.mean())}: ")
logger.info(f"  Mean train-to-test metric: {tr2te_cl:.6f}")
logger.info(f"  Mean RMSE to quantile up metric: {rmse2qu_cl:.6f}")
logger.info(f"  Mean RMSE to quantile down metric: {rmse2qd_cl:.6f}")
logger.info(f"  Mean silhouette to quantile up metric: {sil2qu_cl:.6f}")
logger.info(f"  Mean silhouette to quantile down metric: {sil2qd_cl:.6f}")
logger.info(f"  Mean silhouette to test metric: {sil2te_cl:.6f}")
logger.info(f"  Mean silhouette min to test metric: {silmin2te_cl:.6f}")
logger.info(f"  Mean silhouette min to quantile up metric: {silmin2qu_cl:.6f}")
logger.info(f"  Mean silhouette min to quantile down metric: {silmin2qd_cl:.6f}")